# Multi-Class DDoS Classification for NFV Orchestration

**ProDDoS-NFV Framework — Layer 2: AI Detection Engine**

This notebook extends the binary DDoS detection pipeline to:
1. **Multi-class classification** — Identify specific attack types (12+ classes)
2. **Confidence scoring** — Output per-class probabilities for orchestration decisions
3. **Early prediction** — Detect attacks using first-K packets (truncated flows)
4. **Model export** — Save model as REST-API-ready artifact for orchestration integration

Dataset: CIC-DDoS 2019 (~30 GB)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 1: Imports and Configuration
# ─────────────────────────────────────────────────────────────
import os, glob, gc, warnings, json, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.feature_selection import VarianceThreshold
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, precision_recall_fscore_support,
    roc_auc_score
)
import lightgbm as lgb
import joblib

# ── Paths ──────────────────────────────────────────────────────
BASE_DIR   = Path(r"d:\Khóa luận\Dataset")
MODEL_DIR  = Path(r"d:\Khóa luận\Src\models")
MODEL_DIR.mkdir(exist_ok=True)

CSV_DIRS   = [
    BASE_DIR / "CSV-01-12" / "01-12",
    BASE_DIR / "CSV-03-11" / "03-11",
]

all_csv = []
for d in CSV_DIRS:
    all_csv.extend(sorted(d.glob("*.csv")))

# ── Config ─────────────────────────────────────────────────────
CHUNK_SIZE  = 200_000
SAMPLE_FRAC = 0.05
LABEL_COL   = "Label"
RANDOM_STATE = 42

print(f"Total CSV files found: {len(all_csv)}")
for f in all_csv:
    size_mb = f.stat().st_size / 1e6
    print(f"  {f.name:35s}  {size_mb:8.1f} MB")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 2: Memory-Efficient Data Loading (same as base pipeline)
# ─────────────────────────────────────────────────────────────
def reduce_mem(df: pd.DataFrame) -> pd.DataFrame:
    """Reduce memory usage by downcasting numeric types."""
    for col in df.select_dtypes(include=["float64"]).columns:
        df[col] = df[col].astype("float32")
    for col in df.select_dtypes(include=["int64"]).columns:
        df[col] = df[col].astype("int32")
    return df

sample_frames = []
class_counts = {}  # track per-class sample counts

for csv_path in all_csv:
    local_samples = []
    reader = pd.read_csv(
        csv_path, chunksize=CHUNK_SIZE,
        low_memory=False, encoding="utf-8", on_bad_lines="skip"
    )
    for chunk in reader:
        chunk.columns = chunk.columns.str.strip()
        chunk = reduce_mem(chunk)
        chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
        chunk.dropna(how="all", axis=1, inplace=True)
        
        # Track class distribution
        if LABEL_COL in chunk.columns:
            for label, cnt in chunk[LABEL_COL].value_counts().items():
                label = str(label).strip()
                class_counts[label] = class_counts.get(label, 0) + cnt
        
        n_sample = max(1, int(len(chunk) * SAMPLE_FRAC))
        local_samples.append(chunk.sample(n=n_sample, random_state=RANDOM_STATE))
        del chunk
        gc.collect()
    
    sample_frames.append(pd.concat(local_samples, ignore_index=True))
    print(f"  ✓ {csv_path.name}")
    del local_samples
    gc.collect()

eda_df = pd.concat(sample_frames, ignore_index=True)
del sample_frames
gc.collect()

print(f"\nEDA sample size: {len(eda_df):,} rows")
print(f"RAM: {eda_df.memory_usage(deep=True).sum()/1e6:.1f} MB")
print(f"\nFull dataset class distribution:")
for label, cnt in sorted(class_counts.items(), key=lambda x: -x[1]):
    print(f"  {label:30s} {cnt:>12,}")

## 1. Multi-Class Preprocessing
Unlike the binary pipeline, we keep the **original attack type labels** and encode them as integers.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 3: Preprocessing — Multi-Class Labels
# ─────────────────────────────────────────────────────────────
df = eda_df.copy()

# Drop metadata columns
COLS_TO_DROP = [
    "Flow ID", "Source IP", "Destination IP",
    "Source Port", "Destination Port", "Timestamp",
    "SimillarHTTP",  # sometimes present, non-numeric
]
df.drop(columns=[c for c in COLS_TO_DROP if c in df.columns],
        inplace=True, errors="ignore")

# Clean Inf / NaN
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.fillna(df.median(numeric_only=True), inplace=True)

# Strip label whitespace and normalize
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip()

# Encode multi-class labels
le = LabelEncoder()
df["target"] = le.fit_transform(df[LABEL_COL])
class_names = le.classes_.tolist()
n_classes = len(class_names)

print(f"Number of classes: {n_classes}")
print(f"\nClass mapping:")
for idx, name in enumerate(class_names):
    count = (df["target"] == idx).sum()
    print(f"  {idx:3d} → {name:30s}  ({count:>8,} samples)")

df.drop(columns=[LABEL_COL], inplace=True)

# Save label encoder for later use
joblib.dump(le, MODEL_DIR / "label_encoder.pkl")
print(f"\nLabel encoder saved to {MODEL_DIR / 'label_encoder.pkl'}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 4: Feature Selection
# ─────────────────────────────────────────────────────────────

# Remove zero-variance features
feat_cols = df.select_dtypes(include=[np.number]).columns.drop("target", errors="ignore")
vt = VarianceThreshold(threshold=0.0)
vt.fit(df[feat_cols])
low_var = feat_cols[~vt.get_support()].tolist()
print(f"Dropping {len(low_var)} zero-variance columns")
df.drop(columns=low_var, inplace=True, errors="ignore")

# Remove highly correlated features (|r| > 0.95)
feat_cols = df.select_dtypes(include=[np.number]).columns.drop("target", errors="ignore")
corr_mat = df[feat_cols].corr().abs()
upper = corr_mat.where(np.triu(np.ones(corr_mat.shape), k=1).astype(bool))
to_drop = [c for c in upper.columns if any(upper[c] > 0.95)]
print(f"Dropping {len(to_drop)} highly-correlated features")
df.drop(columns=to_drop, inplace=True, errors="ignore")

feat_cols = [c for c in df.columns if c != "target"]
print(f"\nFinal feature count: {len(feat_cols)}")
print(f"Features: {feat_cols}")

## 2. Multi-Class LightGBM Training

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 5: Train/Test Split + Multi-Class LightGBM
# ─────────────────────────────────────────────────────────────
X = df[feat_cols].values.astype("float32")
y = df["target"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save scaler
joblib.dump(scaler, MODEL_DIR / "multiclass_scaler.pkl")

print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Classes in train: {np.unique(y_train)}")
print(f"Classes in test:  {np.unique(y_test)}")

# ── LightGBM Multi-class ─────────────────────────────────────
lgb_params = {
    "objective": "multiclass",
    "num_class": n_classes,
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 50,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "n_jobs": -1,
    "verbose": -1,
    "is_unbalance": True,  # handle imbalanced classes
}

dtrain = lgb.Dataset(X_train, label=y_train)
dval = lgb.Dataset(X_test, label=y_test, reference=dtrain)

print("\nTraining LightGBM multi-class model...")
lgb_model = lgb.train(
    lgb_params,
    dtrain,
    num_boost_round=300,
    valid_sets=[dval],
    callbacks=[
        lgb.log_evaluation(period=50),
        lgb.early_stopping(stopping_rounds=30),
    ],
)

# Save model
lgb_model.save_model(str(MODEL_DIR / "multiclass_lgb.txt"))
print(f"\nModel saved to {MODEL_DIR / 'multiclass_lgb.txt'}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 6: Multi-Class Evaluation
# ─────────────────────────────────────────────────────────────
y_prob = lgb_model.predict(X_test)  # shape: (n_samples, n_classes)
y_pred = np.argmax(y_prob, axis=1)

print("=== Multi-Class Classification Report ===")
print(classification_report(y_test, y_pred, target_names=class_names, digits=4))

# Per-class metrics
precision, recall, f1, support = precision_recall_fscore_support(
    y_test, y_pred, average=None
)
metrics_df = pd.DataFrame({
    "Class": class_names,
    "Precision": precision,
    "Recall": recall,
    "F1-Score": f1,
    "Support": support,
}).sort_values("F1-Score", ascending=False)

print("\n=== Per-Class F1 Scores ===")
print(metrics_df.to_string(index=False))

# Weighted F1
weighted_f1 = f1_score(y_test, y_pred, average="weighted")
macro_f1 = f1_score(y_test, y_pred, average="macro")
print(f"\nWeighted F1: {weighted_f1:.4f}")
print(f"Macro F1:    {macro_f1:.4f}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 7: Confusion Matrix Visualization
# ─────────────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
cm_normalized = cm.astype("float") / cm.sum(axis=1)[:, np.newaxis]

fig, axes = plt.subplots(1, 2, figsize=(24, 10))

# Raw counts
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=class_names, yticklabels=class_names, ax=axes[0])
axes[0].set_title("Confusion Matrix (Counts)")
axes[0].set_ylabel("True Label")
axes[0].set_xlabel("Predicted Label")
plt.setp(axes[0].get_xticklabels(), rotation=45, ha="right")
plt.setp(axes[0].get_yticklabels(), rotation=0)

# Normalized
sns.heatmap(cm_normalized, annot=True, fmt=".2f", cmap="YlOrRd",
            xticklabels=class_names, yticklabels=class_names, ax=axes[1])
axes[1].set_title("Confusion Matrix (Normalized)")
axes[1].set_ylabel("True Label")
axes[1].set_xlabel("Predicted Label")
plt.setp(axes[1].get_xticklabels(), rotation=45, ha="right")
plt.setp(axes[1].get_yticklabels(), rotation=0)

plt.tight_layout()
plt.savefig(MODEL_DIR / "confusion_matrix_multiclass.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 8: Feature Importance for Multi-Class
# ─────────────────────────────────────────────────────────────
importance = lgb_model.feature_importance(importance_type="gain")
feat_importance = pd.DataFrame({
    "Feature": feat_cols,
    "Importance": importance
}).sort_values("Importance", ascending=False)

# Top 30 features for full-data training
TOP_N = 30
selected_features = feat_importance.head(TOP_N)["Feature"].tolist()

plt.figure(figsize=(12, 8))
feat_importance.head(TOP_N).plot(
    x="Feature", y="Importance", kind="barh", color="steelblue",
    legend=False, ax=plt.gca()
)
plt.title(f"Top {TOP_N} Feature Importances — Multi-Class LightGBM")
plt.xlabel("Gain")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(MODEL_DIR / "feature_importance_multiclass.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"\nSelected top {TOP_N} features:")
print(selected_features)

# Save feature list
joblib.dump(selected_features, MODEL_DIR / "selected_features.pkl")
joblib.dump(feat_cols, MODEL_DIR / "all_features.pkl")

## 3. Early Prediction Experiment
Can we detect the attack type using only **partial flow information**?

We simulate early prediction by progressively masking features that require complete flow observation (e.g., backward packet stats, flow duration) and evaluating degradation.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 9: Early Prediction — Feature Subsets
# ─────────────────────────────────────────────────────────────
# Categorize features by what's available at different stages
# of a flow's lifecycle:
#   - EARLY: available after first few packets (forward-only stats)
#   - MID:   available mid-flow (some backward stats)
#   - FULL:  requires complete flow (duration, total counts)

early_keywords = [
    "fwd", "Fwd", "Init", "SYN", "syn", "Protocol",
    "Src", "Dst", "Header", "PSH", "ACK",
]
late_keywords = [
    "Bwd", "bwd", "Duration", "duration", "Idle", "Active",
    "Down/Up", "Flow Bytes", "Flow Packets",
]

def categorize_feature(feat_name):
    feat_lower = feat_name.lower()
    if any(kw.lower() in feat_lower for kw in late_keywords):
        return "LATE"
    if any(kw.lower() in feat_lower for kw in early_keywords):
        return "EARLY"
    return "MID"

feature_stages = {f: categorize_feature(f) for f in feat_cols}
early_feats = [f for f, s in feature_stages.items() if s == "EARLY"]
mid_feats = [f for f, s in feature_stages.items() if s in ("EARLY", "MID")]
full_feats = feat_cols  # all features

print(f"EARLY features ({len(early_feats)}): {early_feats[:10]}...")
print(f"MID features   ({len(mid_feats)}): first 10: {mid_feats[:10]}...")
print(f"FULL features  ({len(full_feats)})")

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 10: Train & Evaluate at Each Stage
# ─────────────────────────────────────────────────────────────
stage_results = {}

for stage_name, stage_feats in [("EARLY", early_feats), ("MID", mid_feats), ("FULL", full_feats)]:
    if len(stage_feats) == 0:
        print(f"Skipping {stage_name}: no features")
        continue
    
    # Get feature indices
    feat_indices = [feat_cols.index(f) for f in stage_feats if f in feat_cols]
    
    X_tr = X_train[:, feat_indices]
    X_te = X_test[:, feat_indices]
    
    dtrain_s = lgb.Dataset(X_tr, label=y_train)
    dval_s = lgb.Dataset(X_te, label=y_test, reference=dtrain_s)
    
    params_s = lgb_params.copy()
    model_s = lgb.train(
        params_s, dtrain_s,
        num_boost_round=200,
        valid_sets=[dval_s],
        callbacks=[
            lgb.log_evaluation(period=100),
            lgb.early_stopping(stopping_rounds=20),
        ],
    )
    
    y_prob_s = model_s.predict(X_te)
    y_pred_s = np.argmax(y_prob_s, axis=1)
    
    w_f1 = f1_score(y_test, y_pred_s, average="weighted")
    m_f1 = f1_score(y_test, y_pred_s, average="macro")
    
    stage_results[stage_name] = {
        "n_features": len(stage_feats),
        "weighted_f1": w_f1,
        "macro_f1": m_f1,
    }
    print(f"\n{'='*50}")
    print(f"{stage_name} ({len(stage_feats)} features): Weighted-F1={w_f1:.4f}, Macro-F1={m_f1:.4f}")
    
    # Save early model for orchestration (fast inference)
    if stage_name == "EARLY":
        model_s.save_model(str(MODEL_DIR / "early_detection_lgb.txt"))
        joblib.dump(stage_feats, MODEL_DIR / "early_features.pkl")
        print(f"Early detection model saved.")

print("\n=== Early Prediction Summary ===")
results_ep = pd.DataFrame(stage_results).T
print(results_ep)

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 11: Early Prediction Visualization
# ─────────────────────────────────────────────────────────────
if stage_results:
    fig, ax = plt.subplots(figsize=(10, 5))
    stages = list(stage_results.keys())
    x = np.arange(len(stages))
    width = 0.3
    
    w_f1s = [stage_results[s]["weighted_f1"] for s in stages]
    m_f1s = [stage_results[s]["macro_f1"] for s in stages]
    n_feats = [stage_results[s]["n_features"] for s in stages]
    
    bars1 = ax.bar(x - width/2, w_f1s, width, label="Weighted F1", color="steelblue")
    bars2 = ax.bar(x + width/2, m_f1s, width, label="Macro F1", color="coral")
    
    ax.set_xticks(x)
    ax.set_xticklabels([f"{s}\n({n} feats)" for s, n in zip(stages, n_feats)])
    ax.set_ylabel("F1 Score")
    ax.set_title("Early Prediction: F1 Score by Feature Stage")
    ax.legend()
    ax.set_ylim(0, 1.05)
    
    for bar in bars1:
        ax.annotate(f"{bar.get_height():.3f}",
                    xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha="center", va="bottom", fontsize=9)
    for bar in bars2:
        ax.annotate(f"{bar.get_height():.3f}",
                    xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                    ha="center", va="bottom", fontsize=9)
    
    plt.tight_layout()
    plt.savefig(MODEL_DIR / "early_prediction_comparison.png", dpi=150, bbox_inches="tight")
    plt.show()

## 4. Confidence Scoring for Orchestration
The orchestrator needs not just the predicted class, but a **confidence score** to decide action urgency.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 12: Confidence Analysis
# ─────────────────────────────────────────────────────────────
y_prob_full = lgb_model.predict(X_test)
y_pred_full = np.argmax(y_prob_full, axis=1)
y_conf_full = np.max(y_prob_full, axis=1)  # max probability = confidence

# Confidence distribution: correct vs incorrect predictions
correct_mask = y_pred_full == y_test

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confidence histogram
axes[0].hist(y_conf_full[correct_mask], bins=50, alpha=0.7, label="Correct", color="green", density=True)
axes[0].hist(y_conf_full[~correct_mask], bins=50, alpha=0.7, label="Incorrect", color="red", density=True)
axes[0].set_xlabel("Confidence (max probability)")
axes[0].set_ylabel("Density")
axes[0].set_title("Confidence Distribution")
axes[0].legend()

# Accuracy vs confidence threshold
thresholds = np.arange(0.3, 1.0, 0.05)
accs = []
coverages = []
for t in thresholds:
    mask = y_conf_full >= t
    if mask.sum() > 0:
        accs.append((y_pred_full[mask] == y_test[mask]).mean())
        coverages.append(mask.mean())
    else:
        accs.append(np.nan)
        coverages.append(0)

ax2 = axes[1]
ax2.plot(thresholds, accs, "b-o", label="Accuracy")
ax2_twin = ax2.twinx()
ax2_twin.plot(thresholds, coverages, "r--s", label="Coverage")
ax2.set_xlabel("Confidence Threshold")
ax2.set_ylabel("Accuracy", color="blue")
ax2_twin.set_ylabel("Coverage (% of samples)", color="red")
ax2.set_title("Accuracy vs Coverage at Different Confidence Thresholds")
ax2.legend(loc="lower left")
ax2_twin.legend(loc="lower right")

plt.tight_layout()
plt.savefig(MODEL_DIR / "confidence_analysis.png", dpi=150, bbox_inches="tight")
plt.show()

# Determine optimal threshold for orchestration
# We want high accuracy (>0.95) with reasonable coverage
for t, acc, cov in zip(thresholds, accs, coverages):
    if acc and acc >= 0.95 and cov >= 0.5:
        print(f"\nRecommended orchestration threshold: {t:.2f}")
        print(f"  → Accuracy: {acc:.4f}, Coverage: {cov:.4f}")
        break

## 5. Full-Data Multi-Class Training (LightGBM Continued)
Train on the entire 30 GB dataset using checkpoint-based continued training.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 13: Full-Data Multi-Class LightGBM
# ─────────────────────────────────────────────────────────────
N_ROUNDS = 50
FULL_LGB_PARAMS = {
    "objective": "multiclass",
    "num_class": n_classes,
    "metric": "multi_logloss",
    "learning_rate": 0.05,
    "num_leaves": 63,
    "min_child_samples": 50,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "is_unbalance": True,
    "n_jobs": -1,
    "verbose": -1,
}

COLS_TO_DROP_FULL = [
    "Flow ID", "Source IP", "Destination IP",
    "Source Port", "Destination Port", "Timestamp",
    "SimillarHTTP",
]

full_model_path = str(MODEL_DIR / "multiclass_lgb_full.txt")

for file_idx, csv_path in enumerate(all_csv):
    file_chunks = []
    for chunk in pd.read_csv(csv_path, chunksize=CHUNK_SIZE,
                              low_memory=False, on_bad_lines="skip"):
        chunk.columns = chunk.columns.str.strip()
        chunk = reduce_mem(chunk)
        chunk.replace([np.inf, -np.inf], np.nan, inplace=True)
        chunk.drop(columns=[c for c in COLS_TO_DROP_FULL if c in chunk.columns],
                   inplace=True, errors="ignore")
        avail = [f for f in selected_features if f in chunk.columns]
        chunk.fillna(chunk.median(numeric_only=True), inplace=True)
        if LABEL_COL in chunk.columns:
            chunk[LABEL_COL] = chunk[LABEL_COL].astype(str).str.strip()
            file_chunks.append(chunk[avail + [LABEL_COL]])
    
    if not file_chunks:
        continue
    
    file_df = pd.concat(file_chunks, ignore_index=True)
    del file_chunks; gc.collect()
    
    avail = [f for f in selected_features if f in file_df.columns]
    X_file = file_df[avail].values.astype("float32")
    y_file = le.transform(file_df[LABEL_COL].values)
    del file_df; gc.collect()
    
    dtrain = lgb.Dataset(X_file, label=y_file, free_raw_data=True)
    del X_file, y_file; gc.collect()
    
    lgb.train(
        FULL_LGB_PARAMS, dtrain,
        num_boost_round=N_ROUNDS,
        init_model=full_model_path if file_idx > 0 else None,
        keep_training_booster=False,
    ).save_model(full_model_path)
    
    print(f"  [{file_idx+1}/{len(all_csv)}] Trained on {csv_path.name}")

print(f"\nFull multi-class model saved to {full_model_path}")

## 6. Export Model Metadata for Orchestration API
Package all artifacts needed by the REST API server.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 14: Export Model Metadata
# ─────────────────────────────────────────────────────────────
model_metadata = {
    "model_type": "LightGBM",
    "task": "multiclass",
    "n_classes": n_classes,
    "class_names": class_names,
    "selected_features": selected_features,
    "all_features": feat_cols,
    "model_path": "multiclass_lgb.txt",
    "full_model_path": "multiclass_lgb_full.txt",
    "early_model_path": "early_detection_lgb.txt",
    "scaler_path": "multiclass_scaler.pkl",
    "label_encoder_path": "label_encoder.pkl",
    "confidence_threshold": 0.7,  # default, tune based on Cell 12 results
    "attack_type_mapping": {
        name: {
            "class_id": idx,
            "category": "benign" if name == "BENIGN" else "volumetric"
                if any(k in name for k in ["DNS", "NTP", "SSDP", "SNMP", "UDP", "LDAP", "MSSQL", "NetBIOS", "TFTP"])
                else "protocol" if "Syn" in name or "SYN" in name
                else "other",
        }
        for idx, name in enumerate(class_names)
    },
}

with open(MODEL_DIR / "model_metadata.json", "w") as f:
    json.dump(model_metadata, f, indent=2)

print("Model metadata saved to model_metadata.json")
print(json.dumps(model_metadata, indent=2))

In [ ]:
# ─────────────────────────────────────────────────────────────
# Cell 15: Summary
# ─────────────────────────────────────────────────────────────
print("="*60)
print("ProDDoS-NFV — AI Detection Engine Summary")
print("="*60)
print(f"\nDataset: CIC-DDoS 2019 ({len(all_csv)} files)")
print(f"Classes: {n_classes} ({', '.join(class_names)})")
print(f"Features: {len(feat_cols)} total, {TOP_N} selected")
print(f"\nSample Results (5% sample):")
print(f"  Full-feature Weighted F1:  {weighted_f1:.4f}")
print(f"  Full-feature Macro F1:     {macro_f1:.4f}")
if stage_results:
    for stage, res in stage_results.items():
        print(f"  {stage:5s} stage ({res['n_features']:2d} feats): "
              f"W-F1={res['weighted_f1']:.4f}, M-F1={res['macro_f1']:.4f}")
print(f"\nArtifacts saved to: {MODEL_DIR}")
print(f"  - multiclass_lgb.txt (sample-trained model)")
print(f"  - multiclass_lgb_full.txt (full-data model)")
print(f"  - early_detection_lgb.txt (early prediction model)")
print(f"  - multiclass_scaler.pkl")
print(f"  - label_encoder.pkl")
print(f"  - model_metadata.json")
print(f"  - selected_features.pkl")
print(f"\nNext step: Start orchestration API server")
print(f"  → python orchestration/api_server.py")